# Балансування датасету по темах
Вибірка по 6000 записів кожної теми без дублікатів

In [28]:
import pandas as pd
import numpy as np

# Завантаження датасету
df = pd.read_csv('../Datasets/ua_news_train.csv')
print(f"Загальна кількість записів: {len(df)}")
print(f"\nКолонки датасету: {list(df.columns)}")
print(f"\nПерші рядки:")
df.head()

Загальна кількість записів: 120417

Колонки датасету: ['title', 'text', 'tags', 'target']

Перші рядки:


,title,text,tags,target
0,"ВЕРНИДУБ: «Наносимо 25 ударів, 15 у ворота, а ...",Головний тренер солігорського «Шахтаря» Юрій В...,Футбол|Інші новини|Шахтар Солігорськ|Юрій Верн...,спорт
1,"У ""Київстар"" заявляють, що їх обшукала ДФС",Про це на своїй сторінці у Facebook написав пр...,|Київстар|Новини|податкова|ДФС|обшук|,новини
2,В 2016 році 1% найзаможніших людей вперше ста...,Про це повідомляється в доповіді некомерційної...,|багатство|найбагатші люди світу|Новини|бідність|,новини
3,Шакіл О’Ніл продав шикарний маєток у Флориді в...,Легенда НБА Шакіл О’Ніл продав свій маєток у ...,США|Баскетбол|Флорида|Фото,спорт
4,"У «заблокованій» СБУ B2B Jewelry заявили, що п...",Засновник фінансової піраміди B2B Jewelry Мик...,Служба безпеки України (СБУ)|Финансовая пирами...,бізнес


In [29]:
# Перевірка розподілу по темах
print("Розподіл по темах:")
print(df['target'].value_counts())

Розподіл по темах:
target
політика      40364
спорт         28438
новини        25209
бізнес        14759
технології    11647
Name: count, dtype: int64


In [30]:
# Видалення дублікатів
df_no_duplicates = df.drop_duplicates()
print(f"Кількість записів без дублікатів: {len(df_no_duplicates)}")
print(f"Видалено дублікатів: {len(df) - len(df_no_duplicates)}")

print("\nРозподіл по темах після видалення дублікатів:")
print(df_no_duplicates['target'].value_counts())

Кількість записів без дублікатів: 119738
Видалено дублікатів: 679

Розподіл по темах після видалення дублікатів:
target
політика      40248
спорт         28334
новини        25034
бізнес        14678
технології    11444
Name: count, dtype: int64


In [42]:
# Вибірка по 6000 записів кожної теми
n_samples = 10000

# Список для збереження вибраних даних
balanced_dfs = []

# Отримання унікальних тем
targets = df_no_duplicates['target'].unique()
print(f"Знайдені теми: {targets}")

# Вибір по 6000 записів з кожної теми
for target in targets:
    target_df = df_no_duplicates[df_no_duplicates['target'] == target]
    print(f"\nТема '{target}': {len(target_df)} записів")
    
    if len(target_df) >= n_samples:
        # Випадкова вибірка 6000 записів
        sampled_df = target_df.sample(n=n_samples)
        print(f"Вибрано: {len(sampled_df)} записів")
    else:
        # Якщо записів менше 6000, беремо всі
        sampled_df = target_df
        print(f"Вибрано всі доступні: {len(sampled_df)} записів (менше ніж {n_samples})")
    
    balanced_dfs.append(sampled_df)

# Об'єднання всіх вибірок
balanced_df = pd.concat(balanced_dfs, ignore_index=True)

print(f"\n{'='*50}")
print(f"Загальна кількість записів у збалансованому датасеті: {len(balanced_df)}")
print(f"\nРозподіл по темах у збалансованому датасеті:")
print(balanced_df['target'].value_counts())

Знайдені теми: ['спорт' 'новини' 'бізнес' 'політика' 'технології']

Тема 'спорт': 28334 записів
Вибрано: 10000 записів

Тема 'новини': 25034 записів
Вибрано: 10000 записів

Тема 'бізнес': 14678 записів
Вибрано: 10000 записів

Тема 'політика': 40248 записів
Вибрано: 10000 записів

Тема 'технології': 11444 записів
Вибрано: 10000 записів

Загальна кількість записів у збалансованому датасеті: 50000

Розподіл по темах у збалансованому датасеті:
target
спорт         10000
новини        10000
бізнес        10000
політика      10000
технології    10000
Name: count, dtype: int64


In [43]:
# Перемішування датасету
balanced_df_shuffled = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Датасет перемішано")
balanced_df_shuffled.head()

Датасет перемішано


,title,text,tags,target
0,Бійка на Драгобраті: нардепи взяли на поруки ...,Про це повідомляє Еспресо.TV з посиланням на п...,|Правий сектор|поруки|Новини|бійка|Драгобрат|,політика
1,ЯСТРЕМСЬКА: «Не вдалося добре виспатися перед ...,Українська тенісистка Даяна Ястремська розпові...,Теніс|Wta|Вероніка Кудерметова|Даяна Ястремськ...,спорт
2,Джеймі Малларкі – Хама Ворті. Відео нокауту,В ніч з 27 на 28 березня в Лас-Вегасі проходит...,Mma|Ufc|Ufc 260|Джеймі Малларкі|Хама Ворті|від...,спорт
3,Дівчина закохалася в ляльку-зомбі і планує з ...,Про це пише видання Metro. Дівчина продемонстр...,|психологія|курйози|треш|Новини|кохання|,новини
4,"Клінтон стверджує, що Трамп збрехав майже 60 ...","Про це вона написала в Twitter. ""Дональд Трамп...",|США|Дональд Трамп|дебати|Новини|Гілларі Клінтон|,політика


In [44]:
len(balanced_df_shuffled)

50000

In [45]:
# Збереження збалансованого датасету
output_path = '../Datasets/ua_news_train_balanced.csv'
balanced_df_shuffled.to_csv(output_path, index=False)
print(f"Збалансований датасет збережено у: {output_path}")

Збалансований датасет збережено у: ../Datasets/ua_news_train_balanced.csv


In [41]:
# Фінальна статистика
print("="*60)
print("ФІНАЛЬНА СТАТИСТИКА")
print("="*60)
print(f"Вихідний датасет: {len(df)} записів")
print(f"Після видалення дублікатів: {len(df_no_duplicates)} записів")
print(f"Збалансований датасет: {len(balanced_df_shuffled)} записів")
print(f"\nРозподіл по темах:")
for target, count in balanced_df_shuffled['target'].value_counts().items():
    print(f"  {target}: {count} записів")

ФІНАЛЬНА СТАТИСТИКА
Вихідний датасет: 120417 записів
Після видалення дублікатів: 119738 записів
Збалансований датасет: 40000 записів

Розподіл по темах:
  технології: 8000 записів
  бізнес: 8000 записів
  політика: 8000 записів
  спорт: 8000 записів
  новини: 8000 записів
